In [25]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.decomposition import PCA
from unidecode import unidecode
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler


In [ ]:
def build_stats(filename_full='stats_full', filename_reduced='stats_reduced'):
    df = pd.read_csv('unified_player_stats.csv')
    df = df.loc[df['season'] == "2025-26"]
    df = df[['player', '_name_norm', 'team', 'pos', 'games', 'minutes', 'ninety_s', 'shots', 'shots_per90', 'shots_on_target', 'shots_inside_box', 'shots_outside_box', 'npxg', 'npxg_per90', 'npxg_overperformance', 'goal_conversion_pct', 'big_chances_missed', 'key_passes_per90', 'xag_per90', 'attempt_assists', 'big_chances_created_per90', 'xg_chain_per90', 'xg_buildup_per90', 'pass_to_assist', 'passes_total', 'pass_completion_pct', 'passes_final_third', 'passes_opp_half', 'long_balls_total', 'long_balls_pct', 'crosses_total', 'crosses_pct', 'chipped_passes_total', 'chipped_passes_pct', 'touches', 'dribbles_per90', 'dribbles_pct', 'dispossessed', 'possession_lost', 'ball_recoveries', 'possession_won_att_third', 'tackles', 'tackles_won_pct', 'interceptions_per90', 'clearances', 'blocked_shots', 'dribbled_past', 'aerials_won_pct', 'ground_duels_won_pct', 'duels_won_pct', 'fouls', 'fouled']]
    df['shots_on_target_per90'] = df['shots_on_target']/df['ninety_s']
    df['shots_inside_box_per90'] = df['shots_inside_box']/df['ninety_s']
    df['shots_outside_box_per90'] = df['shots_outside_box']/df['ninety_s']
    df['npxg_per_shot'] = df['npxg']/df['shots']
    df["npxg_per_shot"] = df["npxg_per_shot"].fillna(0)
    df['big_chances_missed_per90'] = df['big_chances_missed']/df['ninety_s']
    df['attempt_assists_per90'] = df['attempt_assists']/df['ninety_s']
    df['pass_to_assist_per90'] = df['pass_to_assist']/df['ninety_s']
    df['passes_per90'] = df['passes_total']/df['ninety_s']
    df['passes_final_third_per90'] = df['passes_final_third']/df['ninety_s']
    df['passes_opp_half_per90'] = df['passes_opp_half']/df['ninety_s']
    df['long_balls_per90'] = df['long_balls_total']/df['ninety_s']
    df['crosses_per90'] = df['crosses_total']/df['ninety_s']
    df['chipped_passes_per90'] = df['chipped_passes_total']/df['ninety_s']
    df['touches_per90'] = df['touches']/df['ninety_s']
    df['dispossessed_per90'] = df['dispossessed']/df['ninety_s']
    df['possession_lost_per90'] = df['possession_lost']/df['ninety_s']
    df['ball_recoveries_per90'] = df['ball_recoveries']/df['ninety_s']
    df['possession_won_att_third_per90'] = df['possession_won_att_third']/df['ninety_s']
    df['tackles_per90'] = df['tackles']/df['ninety_s']
    df['clearances_per90'] = df['clearances']/df['ninety_s']
    df['blocked_shots_per90'] = df['blocked_shots']/df['ninety_s']
    df['dribbled_past_per90'] = df['dribbled_past']/df['ninety_s']
    df['fouls_per90'] = df['fouls']/df['ninety_s']
    df['fouled_per90'] = df['fouled']/df['ninety_s']
    df['npxg_overperformance_per90'] = df['npxg_overperformance']/df['ninety_s']
    # df['player_norm'] = df['_name_norm']
    df['player_norm'] = df['player'].apply(unidecode)

    df.to_csv(f"{filename_full}.csv", index=False, header=True)
    df_reduced = df[['player', 'player_norm', 'team', 'pos', 'games', 'minutes', 'ninety_s', 'shots_per90', 'shots_on_target_per90', 'shots_inside_box_per90', 'shots_outside_box_per90', 'npxg_per90','npxg_per_shot', 'npxg_overperformance_per90', 'goal_conversion_pct', 'big_chances_missed_per90', 'key_passes_per90', 'xag_per90', 'attempt_assists_per90', 'big_chances_created_per90', 'xg_chain_per90', 'xg_buildup_per90', 'passes_per90', 'pass_completion_pct', 'passes_final_third_per90', 'passes_opp_half_per90', 'long_balls_per90', 'long_balls_pct', 'crosses_per90', 'crosses_pct', 'chipped_passes_per90', 'chipped_passes_pct', 'touches_per90', 'dribbles_per90', 'dribbles_pct', 'dispossessed_per90', 'possession_lost_per90', 'ball_recoveries_per90', 'possession_won_att_third_per90', 'tackles_per90', 'tackles_won_pct', 'interceptions_per90', 'clearances_per90', 'blocked_shots_per90', 'dribbled_past_per90', 'aerials_won_pct', 'ground_duels_won_pct', 'duels_won_pct', 'fouls_per90', 'fouled_per90']]
    df_reduced.to_csv(f"{filename_reduced}.csv", index=False, header=True)

In [27]:
def build_df(filename='stats_reduced', minutes=1500):

    df = pd.read_csv('stats_reduced.csv')
    df_min = df.loc[df['minutes'] >= minutes]
    df_min = df_min.loc[(df['pos'] != 'GK') & (df['pos'] != 'GK S')]
    df_min = df_min.dropna(subset=['pos'])
    df_min = df_min.reset_index()
    df_min = df_min.drop(columns=['index'])
    df_min['goal_conversion_pct'] = df_min['goal_conversion_pct']/100
    df_min['pass_completion_pct'] = df_min['pass_completion_pct']/100
    df_min['long_balls_pct'] = df_min['long_balls_pct']/100
    df_min['crosses_pct'] = df_min['crosses_pct']/100
    df_min['chipped_passes_pct'] = df_min['chipped_passes_pct']/100
    df_min['dribbles_pct'] = df_min['dribbles_pct']/100
    df_min['tackles_won_pct'] = df_min['tackles_won_pct']/100
    df_min['aerials_won_pct'] = df_min['aerials_won_pct']/100
    df_min['ground_duels_won_pct'] = df_min['ground_duels_won_pct']/100
    df_min['duels_won_pct'] = df_min['duels_won_pct']/100


    return df_min

In [28]:
def standardize_df(df):
    scaler = preprocessing.StandardScaler()
    data_df = df.drop(columns=['player', 'player_norm', 'team', 'pos', 'minutes', 'ninety_s', 'games'])
    standard_df = scaler.fit_transform(data_df)
    standard_df = pd.DataFrame(standard_df, columns=data_df.columns)
    standard_df.insert(0, 'player', df['player'])
    standard_df.insert(0, 'player_norm', df['player_norm'])
    standard_df.insert(2, 'team', df['team'])
    standard_df.insert(3, 'pos', df['pos'])
    standard_df.insert(4, 'minutes', df['minutes'])
    standard_df.insert(5, 'ninety_s', df['ninety_s'])
    standard_df.insert(6, 'games', df['games'])
    return standard_df

In [29]:
def apply_pca(df_standard):
    df_pca = df_standard.drop(columns=['player', 'player_norm', 'team', 'pos', 'minutes', 'ninety_s', 'games'])
    pca = PCA(n_components=7)
    data_pca = pca.fit_transform(df_pca)
    data_pca = pd.DataFrame(data_pca)
    data_pca.insert(0, 'player', df_standard['player'])
    data_pca.insert(1, 'player_norm', df_standard['player_norm'])
    data_pca.insert(2, 'team', df_standard['team'])
    data_pca.insert(3, 'pos', df_standard['pos'])
    data_pca.insert(4, 'minutes', df_standard['minutes'])
    data_pca.insert(5, 'ninety_s', df_standard['ninety_s'])
    data_pca.insert(6, 'games', df_standard['games'])

    cont_df = pd.DataFrame()
    feature_names = df_pca.columns
    cont0 = []
    cont1 = []
    cont2 = []
    cont3 = []
    cont4 = []
    cont5 = []
    cont6 = []
    loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
    for i in range(len(feature_names)):
        # print(f"Feature: {feature_names[i]}, Loadings: {loadings[i]}")
        cont0.append(loadings[i][0])
        cont1.append(loadings[i][1])
        cont2.append(loadings[i][2])
        cont3.append(loadings[i][3])
        cont4.append(loadings[i][4])
        cont5.append(loadings[i][5])
        cont6.append(loadings[i][6])
    cont_df['feature'] = feature_names
    cont_df['cont_0'] = cont0
    cont_df['cont_1'] = cont1
    cont_df['cont_2'] = cont2
    cont_df['cont_3'] = cont3
    cont_df['cont_4'] = cont4
    cont_df['cont_5'] = cont5
    cont_df['cont_6'] = cont6
    # print(f"PCA Explained Variance: {pca.explained_variance_ratio_}")


    loading_matrix = pd.DataFrame(
        pca.components_.T,
        index=feature_names,
        columns=[
            'PC1', 'PC2', 'PC3',
            'PC4', 'PC5', 'PC6', 'PC7'
        ]
    )

    loading_matrix['loading_strength'] = np.sqrt(
        (loading_matrix ** 2).sum(axis=1)
    )

    loading_matrix.sort_values(
        'loading_strength',
        ascending=False
    )
    return data_pca, cont_df, loading_matrix

In [30]:
def get_distances(player_name, df_pca):
    search_row = df_pca.loc[df_pca['player_norm'] == player_name]
    dist_df = pd.DataFrame()
    players = []
    distances = []
    cos_sims = []
    teams = []
    positions = []

    

    for i in tqdm(df_pca.index, "Computing distances"):
        compare_row = df_pca.loc[i]
        search_player = search_row['player_norm']
        compare_player = compare_row['player_norm']
        compare_team = compare_row['team']
        compare_pos = compare_row['pos']
        search_data = search_row[[0, 1, 2, 3, 4, 5, 6]].to_numpy(dtype=float)
        compare_data = compare_row[[0, 1, 2, 3, 4, 5, 6]].to_numpy(dtype=float)
        distance = np.linalg.norm(search_data - compare_data)
        cos_sim = cosine_similarity(search_data.reshape(1, -1), compare_data.reshape(1, -1))[0][0]
        # print(f"Distance between {search_player} and {compare_player}: {distance}")
        players.append(compare_player)
        distances.append(distance)
        cos_sims.append(cos_sim)
        teams.append(compare_team)
        positions.append(compare_pos)

    
    
    dist_df['player'] = players
    dist_df['team'] = teams
    dist_df['position'] = positions
    dist_df['distance'] = distances
    dist_df['cosine_similarity'] = cos_sims
    dist_df = dist_df.sort_values(by=['distance'], ascending=True)
    dist_df = dist_df.reset_index(drop=True)
    dist_df['rank_distance'] = dist_df['distance'].rank(method='min')
    dist_df['rank_cosine_similarity'] = dist_df['cosine_similarity'].rank(ascending=False, method='min')
    dist_df['rank_distance'] = dist_df['rank_distance'].astype(int)
    dist_df['rank_cosine_similarity'] = dist_df['rank_cosine_similarity'].astype(int)

    return dist_df

In [31]:
build_stats()
df = build_df(minutes=1000)
# df_standard = standardize_df(df)
# df_pca, cont_df, loadings = apply_pca(df_standard)
# df_pca.head()
# print(loadings['loading_strength'].sort_values(ascending=False))
df[df["npxg_per_shot"].isna()]

C:\Users\andre\AppData\Local\Temp\ipykernel_10032\3543641953.py:2: DtypeWarning: Columns (0: pos) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('unified_player_stats.csv')


,player,player_norm,team,pos,games,minutes,ninety_s,shots_per90,shots_on_target_per90,shots_inside_box_per90,...,tackles_won_pct,interceptions_per90,clearances_per90,blocked_shots_per90,dribbled_past_per90,aerials_won_pct,ground_duels_won_pct,duels_won_pct,fouls_per90,fouled_per90
70,Lamare Bogarde,Lamare Bogarde,Aston Villa,D M S,28,1011,11.23,0.0,0.000000,0.000000,...,0.6923,0.445,2.048085,0.000000,0.445236,0.3750,0.5647,0.5229,1.513802,1.780944
78,Adam Smith,Adam Smith,Bournemouth,D S,22,1079,11.99,0.0,0.000000,0.000000,...,0.6216,1.751,4.086739,0.083403,1.668057,0.4524,0.5000,0.4880,2.502085,1.834862
315,Aaron Wan-Bissaka,Aaron Wan-Bissaka,West Ham,D M S,25,2093,23.26,0.0,0.214961,0.558899,...,0.7000,2.837,4.772141,0.128977,0.386930,0.4107,0.6178,0.5810,1.074807,1.160791
705,Jordy Makengo,Jordy Makengo,Freiburg,D S,20,1160,12.89,0.0,0.000000,0.000000,...,0.4545,1.164,4.034135,0.000000,0.310318,0.5833,0.6333,0.6190,0.775795,0.698216


In [ ]:
print(cont_df[['feature', 'cont_0']].sort_values(key=abs,by=['cont_0'], ascending=False).head(10))
print(cont_df[['feature', 'cont_1']].sort_values(key=abs,by=['cont_1'], ascending=False).head(10))
print(cont_df[['feature', 'cont_2']].sort_values(key=abs,by=['cont_2'], ascending=False).head(10))
print(cont_df[['feature', 'cont_3']].sort_values(key=abs,by=['cont_3'], ascending=False).head(10))
print(cont_df[['feature', 'cont_4']].sort_values(key=abs,by=['cont_4'], ascending=False).head(10))
print(cont_df[['feature', 'cont_5']].sort_values(key=abs,by=['cont_5'], ascending=False).head(10))
print(cont_df[['feature', 'cont_6']].sort_values(key=abs,by=['cont_6'], ascending=False).head(10))

In [54]:
player_name = "Federico Dimarco"
sim_df = get_distances(player_name, df_pca)
sim_df

RangeIndex(start=0, stop=1423, step=1)


Computing distances: 100%|██████████| 1423/1423 [00:00<00:00, 1435.41it/s]


,player,team,position,distance,cosine_similarity,rank_distance,rank_cosine_similarity
0,Federico Dimarco,Inter,D S,0.000000,1.000000,1,1
1,Bruno Fernandes,Manchester United,M,2.398073,0.956028,2,3
2,Alex Grimaldo,Bayer Leverkusen,D,2.785005,0.969816,3,2
3,Florian Thauvin,Lens,M S,2.863920,0.931641,4,5
4,Mathis Cherki,Manchester City,F M S,3.017899,0.947683,5,4
...,...,...,...,...,...,...,...
1418,Ademola Lookman,"Atalanta,Atletico Madrid",F M S,17.394791,0.586667,1419,164
1419,Angel Gomes,"Marseille,Wolverhampton Wanderers",M S,17.672619,0.373364,1420,319
1420,Xavi Simons,"RasenBallsport Leipzig,Tottenham",F M S,18.456276,0.420164,1421,288
1421,Diego Coppola,"Brighton,Paris FC",D,21.926462,-0.403942,1422,1001


In [12]:
df = pd.read_csv('stats_reduced.csv')
df = df.drop(columns=['player', 'player_norm', 'team', 'pos', 'games', 'minutes', 'ninety_s'])
columns = df.columns
corr = df[columns].corr()
corr_pairs = (
    corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )
    .stack()
    .sort_values(key=abs, ascending=False)
)

print(corr_pairs.head(30))

shots_on_target_per90           shots_inside_box_per90            0.987413
passes_per90                    touches_per90                     0.985855
touches_per90                   ball_recoveries_per90             0.982383
passes_final_third_per90        passes_opp_half_per90             0.981506
passes_per90                    passes_opp_half_per90             0.979599
shots_inside_box_per90          dispossessed_per90                0.975089
shots_on_target_per90           dispossessed_per90                0.973291
passes_final_third_per90        ball_recoveries_per90             0.972539
passes_per90                    ball_recoveries_per90             0.970899
passes_opp_half_per90           touches_per90                     0.969315
passes_final_third_per90        touches_per90                     0.967557
possession_won_att_third_per90  blocked_shots_per90               0.965875
shots_on_target_per90           big_chances_missed_per90          0.963151
xg_chain_per90           

In [8]:
df = pd.read_csv('unified_player_stats.csv')
df

C:\Users\andre\AppData\Local\Temp\ipykernel_10032\3103974788.py:1: DtypeWarning: Columns (0: pos) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('unified_player_stats.csv')


,understat_id,player,team,pos,games,minutes,goals,assists,shots,key_passes,...,xg_chain_per90,xg_buildup_per90,shots_per90,key_passes_per90,tackles_won_per90,interceptions_per90,big_chances_created_per90,dribbles_per90,xg_overperformance,npxg_overperformance
0,11295.0,David Datro Fofana,"1. FC Union Berlin,Burnley,Union Berlin",F S,31,1758,6,1,53,18,...,0.496,0.068,2.714,0.922,0.666,0.051,0.154,1.997,-2.99,-3.09
1,3699.0,Aissa Laidouni,"1. FC Union Berlin,Union Berlin",M S,29,1305,0,4,15,15,...,0.334,0.208,1.034,1.034,1.172,0.897,0.276,1.103,-0.98,-0.77
2,8017.0,Alex Kral,"1. FC Union Berlin,Union Berlin",D M S,30,1587,1,1,21,7,...,0.266,0.173,1.191,0.397,1.021,0.737,0.113,0.340,-0.78,-1.72
3,11236.0,Aljoscha Kemlein,"1. FC Union Berlin,Union Berlin",S,3,36,0,0,1,0,...,1.686,1.604,2.500,0.000,0.000,0.000,0.000,2.500,-0.03,-0.03
4,10751.0,Brenden Aaronson,"1. FC Union Berlin,Union Berlin",F M S,36,1337,2,2,22,22,...,0.500,0.226,1.480,1.480,1.211,0.336,0.269,2.355,-0.86,-0.38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25151,NaN,Maximilian Mittelstädt,VfB Stuttgart,NaN,11,854,1,1,12,18,...,0.000,0.000,1.264,1.897,1.475,1.370,0.421,0.738,-0.22,0.00
25152,NaN,Nikolas Nartey,VfB Stuttgart,NaN,5,126,0,1,3,4,...,0.000,0.000,2.143,2.857,2.143,0.000,0.714,3.571,-0.36,0.00
25153,NaN,Ramon Hendriks,VfB Stuttgart,NaN,12,826,0,0,4,5,...,0.000,0.000,0.436,0.545,0.980,1.198,0.000,0.436,-0.17,0.00
25154,NaN,Tiago Tomás,VfB Stuttgart,NaN,9,476,2,0,17,2,...,0.000,0.000,3.214,0.378,0.945,0.000,0.189,2.079,0.15,0.00
